# SSDSE を pandas で読み込む

統計データ分析コンペティション（高校生の部）に向けて、最初の関門
**「日本語CSVの読み込み」** を突破するためのノートブックです。

SSDSEのCSVでつまずくポイントは大きく **2つ**：

| つまずき | 内容 | 対処 |
|---|---|---|
| ① 文字コード | Shift-JIS（cp932）保存なのにpandasはUTF-8で読もうとする | `encoding="cp932"` |
| ② ヘッダー | 見出しが1行ではなく、先頭に情報行が **3行** ある | 項目名の行を見出しにする |

このノートは上から順にセルを実行していけば、そのまま使えるようになっています。

## 準備

- SSDSE-基本素材(SSDSE-E) を公式サイトからCSVでダウンロードし、このノートと同じフォルダに **`SSDSE-E-2026.csv`** という名前で置いてください。

In [1]:
import pandas as pd
from pathlib import Path

CSV_PATH = "SSDSE-E-2026.csv"

## STEP1　何も考えずに読む → 失敗します

まずは `encoding` を指定せずに読むと**日本語のところでエラーになる**。

In [2]:
try:
    df = pd.read_csv(CSV_PATH)          # encoding指定なし＝UTF-8のつもりで読む
    print(df.head())
except UnicodeDecodeError as e:
    print("→ UnicodeDecodeError が出た！")
    print("  ", e)
    print("  【原因】ファイルはShift-JISなのにUTF-8で読もうとしたため。")

→ UnicodeDecodeError が出た！
   'utf-8' codec can't decode byte 0x94 in position 638: invalid start byte
  【原因】ファイルはShift-JISなのにUTF-8で読もうとしたため。


## STEP2　文字コード（`encoding="cp932"`）

`cp932` はShift-JISを拡張したもの。`shift_jis` でもだいたい読めますが、
**①②や髙などの特殊文字**が含まれると `shift_jis` ではエラーになることがあります。
日本の公的データは **`cp932`** を指定しておくのが安全です。

In [3]:
df = pd.read_csv(CSV_PATH, encoding="cp932")
print("読めた。でも見出しがおかしい（コードや年度が見出しになっている）:")
df.head(3)

読めた。でも見出しがおかしい（コードや年度が見出しになっている）:


,SSDSE-E-2026,Prefecture,A1101,A1102,A1301,A1302,A1303,A1700,A4101,A4103,...,I5103,I6100,I6200,I6300,J2503,J2506,L3221,L322101,L322102,L322109
0,NaN,年度,2024,2024,2024,2024,2024,2020,2023,2023,...,2023,2022,2022,2022,2023,2023,2024,2024,2024,2024
1,地域コード,都道府県,総人口,日本人人口,15歳未満人口,15～64歳人口,65歳以上人口,外国人人口,出生数,合計特殊出生率,...,歯科診療所数,医師数,歯科医師数,薬剤師数,保育所等数,保育所等在所児数,消費支出（二人以上の世帯）,食料費（二人以上の世帯）,住居費（二人以上の世帯）,教養娯楽費（二人以上の世帯）
2,R00000,全国,123802000,120296000,13830000,73728000,36243000,2402460,727288,1.20,...,66818,343275,105267,323690,23725,1905477,300243,85040,18074,29098


## STEP3　いきなり整形せず「生の形」を観察する

**新しいデータに出会ったら、まず生のまま眺める。** これはコンペのどんなデータでも効く一番大事な作法です。
`header=None` を付けると、pandasは「1行目＝見出し」という思い込みをやめ、全部をデータとして読みます。

In [4]:
raw = pd.read_csv(CSV_PATH, encoding="cp932", header=None)
print("0行目=項目コード / 1行目=年次 / 2行目=項目名（これを見出しにしたい）/ 3行目〜=データ")
raw.head(5)

0行目=項目コード / 1行目=年次 / 2行目=項目名（これを見出しにしたい）/ 3行目〜=データ


,0,1,2,3,4,5,6,7,8,9,...,85,86,87,88,89,90,91,92,93,94
0,SSDSE-E-2026,Prefecture,A1101,A1102,A1301,A1302,A1303,A1700,A4101,A4103,...,I5103,I6100,I6200,I6300,J2503,J2506,L3221,L322101,L322102,L322109
1,NaN,年度,2024,2024,2024,2024,2024,2020,2023,2023,...,2023,2022,2022,2022,2023,2023,2024,2024,2024,2024
2,地域コード,都道府県,総人口,日本人人口,15歳未満人口,15～64歳人口,65歳以上人口,外国人人口,出生数,合計特殊出生率,...,歯科診療所数,医師数,歯科医師数,薬剤師数,保育所等数,保育所等在所児数,消費支出（二人以上の世帯）,食料費（二人以上の世帯）,住居費（二人以上の世帯）,教養娯楽費（二人以上の世帯）
3,R00000,全国,123802000,120296000,13830000,73728000,36243000,2402460,727288,1.20,...,66818,343275,105267,323690,23725,1905477,300243,85040,18074,29098
4,R01000,北海道,5043000,4980000,497000,2868000,1677000,34321,24430,1.06,...,2742,13613,4302,11625,790,48159,285502,79471,25214,26635


## STEP4　きれいなDataFrameを作る（実用の読み込み関数）

**「地域コード」で始まる行**を探し、そこを見出し行にします。
`header=その行番号` を指定すると、それより上の行（コード行・年次行）は自動でスキップされます。
この方法なら **SSDSE-A〜Eでヘッダー行数が違っても壊れにくい**です。

In [5]:
def load_ssdse(path, encoding="cp932"):
    """SSDSEのCSVを読み、日本語の項目名を列見出しにして返す"""
    raw = pd.read_csv(path, encoding=encoding, header=None, dtype=str)
    name_row = raw.index[raw[0] == "地域コード"][0]   # 項目名の行番号を探す
    return pd.read_csv(path, encoding=encoding, header=name_row)

df = load_ssdse(CSV_PATH)
print("列名:", list(df.columns))
print("形状 (行数, 列数):", df.shape)
print("総人口 の型:", df["総人口"].dtype, "（int64ならOK。objectだと計算できない）")
df.head()

列名: ['地域コード', '都道府県', '総人口', '日本人人口', '15歳未満人口', '15～64歳人口', '65歳以上人口', '外国人人口', '出生数', '合計特殊出生率', '死亡数', '転入者数（日本人移動者）', '転出者数（日本人移動者）', '一般世帯数', '一般世帯人員数', '単独世帯数', '婚姻件数', '離婚件数', '総面積（北方地域及び竹島を除く）', '可住地面積', '自然公園面積', '県内総生産額（平成27年基準）', '県民所得（平成27年基準）', '1人当たり県民所得（平成27年基準）', '事業所数（民営）', '事業所数（民営）（建設業）', '事業所数（民営）（製造業）', '事業所数（民営）（情報通信業）', '事業所数（民営）（卸売業、小売業）', '事業所数（民営）（宿泊業、飲食サービス業）', '事業所数（民営）（生活関連サービス業、娯楽業）', '事業所数（民営）（医療、福祉）', '従業者数（民営）', '従業者数（民営）（建設業）', '従業者数（民営）（製造業）', '従業者数（民営）（情報通信業）', '従業者数（民営）（卸売業、小売業）', '従業者数（民営）（宿泊業、飲食サービス業）', '従業者数（民営）（生活関連サービス業、娯楽業）', '従業者数（民営）（医療、福祉）', '農家数（販売農家）', '農家数（自給的農家）', '耕地面積', '旅館営業施設数（ホテルを含む）', '旅館営業施設客室数（ホテルを含む）', '幼稚園数', '幼稚園在園者数', '小学校数', '小学校児童数', '中学校数', '中学校生徒数', '義務教育学校数', '義務教育学校前期課程児童数', '義務教育学校後期課程生徒数', '高等学校数', '高等学校生徒数', '短期大学数', '大学数', '短期大学学生数', '大学学生数', '公民館数', '図書館数', '博物館数', '劇場、音楽堂等数', '社会体育施設数', '民間体育施設数', '映画館数', '一般旅券発行件数', '延べ宿泊者数', '外国人延べ宿泊者数', '総住宅数', '空き家数', '持ち家数', '一戸建住宅数', '1住宅当たり延べ面積', '総人口（非水洗化人口＋水洗化人口）', '非水洗化

,地域コード,都道府県,総人口,日本人人口,15歳未満人口,15～64歳人口,65歳以上人口,外国人人口,出生数,合計特殊出生率,...,歯科診療所数,医師数,歯科医師数,薬剤師数,保育所等数,保育所等在所児数,消費支出（二人以上の世帯）,食料費（二人以上の世帯）,住居費（二人以上の世帯）,教養娯楽費（二人以上の世帯）
0,R00000,全国,123802000,120296000,13830000,73728000,36243000,2402460,727288,1.20,...,66818,343275,105267,323690,23725,1905477,300243,85040,18074,29098
1,R01000,北海道,5043000,4980000,497000,2868000,1677000,34321,24430,1.06,...,2742,13613,4302,11625,790,48159,285502,79471,25214,26635
2,R02000,青森県,1165000,1157000,114000,634000,416000,5409,5696,1.23,...,481,2795,715,2373,230,12482,260920,80057,12744,21714
3,R03000,岩手県,1145000,1134000,115000,624000,405000,6937,5432,1.16,...,542,2758,965,2572,265,15199,313386,81390,28758,27174
4,R04000,宮城県,2248000,2219000,243000,1340000,665000,19453,12328,1.07,...,1038,6140,1921,5570,411,28610,315200,83820,33163,26160


## STEP5　読み込めたら、いよいよ基本の処理

都道府県を行ラベルにし、「全国」行（47都道府県の合計）を外してから分析するのが定番です。

In [6]:
df = df.set_index("都道府県")          # 都道府県を行ラベルに
pref = df.drop(index="全国")           # 合計の「全国」行は比較のとき外す

print("● 総人口トップ5:")
print(pref["総人口"].sort_values(ascending=False).head(5))

● 総人口トップ5:
都道府県
東京都     14178000
神奈川県     9225000
大阪府      8757000
愛知県      7460000
埼玉県      7332000
Name: 総人口, dtype: int64


In [7]:
# 条件で絞り込む（総人口300万人以上）
big = pref[pref["総人口"] >= 3_000_000]
print(f"● 総人口300万人以上: {len(big)}件 →", big.index.tolist())

# 列どうしの計算で新しい指標を作る（15歳未満人口の割合）
pref = pref.copy()
pref["15歳未満割合(%)"] = pref["15歳未満人口"] / pref["総人口"] * 100
print("\n● 15歳未満割合が高い順トップ3:")
print(pref["15歳未満割合(%)"].round(2).sort_values(ascending=False).head(3))

● 総人口300万人以上: 10件 → ['北海道', '埼玉県', '千葉県', '東京都', '神奈川県', '静岡県', '愛知県', '大阪府', '兵庫県', '福岡県']

● 15歳未満割合が高い順トップ3:
都道府県
沖縄県    15.76
滋賀県    12.77
佐賀県    12.69
Name: 15歳未満割合(%), dtype: float64


In [8]:
# 全体をざっと要約
pref[["総人口", "15歳未満割合(%)"]].describe().round(1)

,総人口,15歳未満割合(%)
count,47.0,47.0
mean,2634106.4,11.2
std,2806597.1,1.1
min,531000.0,8.8
25%,1022000.0,10.6
50%,1532000.0,11.1
75%,2617000.0,11.7
max,14178000.0,15.8


## （発展）項目コード・年次の対応表

論文では「A1101（総人口, 2022年）」のように **出典年**を書く必要があります。
コード行・年次行・項目名行を1枚の対応表にしておくと便利です。

In [9]:
def build_metadata(path, encoding="cp932"):
    raw = pd.read_csv(path, encoding=encoding, header=None, dtype=str)
    meta = pd.DataFrame({"項目コード": raw.iloc[0], "項目名": raw.iloc[2], "年次": raw.iloc[1]})
    return meta.iloc[2:].reset_index(drop=True)   # 先頭2列(地域コード・都道府県)は除く

build_metadata(CSV_PATH)

,項目コード,項目名,年次
0,A1101,総人口,2024
1,A1102,日本人人口,2024
2,A1301,15歳未満人口,2024
3,A1302,15～64歳人口,2024
4,A1303,65歳以上人口,2024
...,...,...,...
88,J2506,保育所等在所児数,2023
89,L3221,消費支出（二人以上の世帯）,2024
90,L322101,食料費（二人以上の世帯）,2024
91,L322102,住居費（二人以上の世帯）,2024
